# Notebook 01 — Initial Data Exploration

**Project:** E-Commerce Customer Value Prediction  
**Dataset:** Online Retail II (UCI)  
**Purpose:** Understand the raw data and validate `problem-formulation.md` assumptions

---

## Purpose

This notebook has **one goal**: understand the raw transaction data.

**Does NOT:**
- Remove rows
- Clean values
- Create features
- Construct the target

All cleaning decisions are **deferred to Notebook 02**.

---

## Proposed Design

| Parameter | Value |
|---|---|
| Prediction date | `2011-09-01` |
| Historical window | `2009-12-01` to `2011-08-31` |
| Future window | `2011-09-01` to last available date |
| Unit of analysis | One row = one customer |
| Target | Future revenue = sum(Quantity x Price) after prediction date |

We **verify** these from actual data — not assume them.

---
## Step 1 — Load the Workbook

Source file: `online_retail_II.xlsx` in `data/`.  
Two sheets: `Year 2009-2010` and `Year 2010-2011`.  
We load separately to verify structure before combining.

In [ ]:
# Step 1.1 — Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)
print(f'pandas {pd.__version__} / numpy {np.__version__}')

In [ ]:
# Step 1.2 — Inspect sheet names
# pd.ExcelFile opens without loading data — efficient for large files.
EXCEL_PATH = '../data/raw/online_retail_II.xlsx'
xl = pd.ExcelFile(EXCEL_PATH)
print('Sheets:', xl.sheet_names)

In [ ]:
# Step 1.3 — Load both sheets separately
# We do NOT force dtypes; we want to see pandas' inferences.
print('Loading Year 2009-2010 ...')
df_0910 = pd.read_excel(EXCEL_PATH, sheet_name='Year 2009-2010')
print(f'  {df_0910.shape[0]:,} rows x {df_0910.shape[1]} cols')
print('Loading Year 2010-2011 ...')
df_1011 = pd.read_excel(EXCEL_PATH, sheet_name='Year 2010-2011')
print(f'  {df_1011.shape[0]:,} rows x {df_1011.shape[1]} cols')

In [ ]:
# Step 1.4 — Compare shapes
print(f'Year 2009-2010 : {df_0910.shape}')
print(f'Year 2010-2011 : {df_1011.shape}')
print(f'Projected combined: {df_0910.shape[0]+df_1011.shape[0]:,} rows')

In [ ]:
# Step 1.5 — Confirm column names identical
print('2009-2010 columns:', df_0910.columns.tolist())
print('2010-2011 columns:', df_1011.columns.tolist())
match = df_0910.columns.tolist() == df_1011.columns.tolist()
print('Columns identical:', match)

---
## Step 2 — Understand Each Column

| Column | Meaning |
|---|---|
| `Invoice` | Order ID. Multiple rows per order (one per product). `C`-prefix = cancellation. |
| `StockCode` | Product SKU. Non-numeric codes are system entries. |
| `Description` | Product name. Should be 1-to-1 with StockCode. |
| `Quantity` | Units on this line. **Negative = return/cancellation.** |
| `InvoiceDate` | Transaction timestamp. Key temporal variable. |
| `Price` | Unit price (GBP). Line revenue = `Quantity x Price`. |
| `Customer ID` | Customer key. **Missing = cannot attribute to customer.** |
| `Country` | Customer country. Will be encoded as a feature. |

In [ ]:
# Step 2.1 — Data types
# Expected: InvoiceDate->datetime64, Quantity->int64, Price->float64,
# Customer ID->float64 (NaN forces int->float), others->object
print('=== dtypes 2009-2010 ==='); print(df_0910.dtypes.to_string())
print(); print('=== dtypes 2010-2011 ==='); print(df_1011.dtypes.to_string())

In [ ]:
# Step 2.2 — Sample rows
print('=== Sample 2009-2010 ==='); display(df_0910.head(10))

In [ ]:
print('=== Sample 2010-2011 ==='); display(df_1011.head(10))

In [ ]:
# Step 2.3 — Missing values per column
def missing_summary(df, label):
    n = len(df); m = df.isnull().sum(); p = (m/n*100).round(2)
    s = pd.DataFrame({'count': m, 'pct%': p})
    print(f'=== Missing: {label} ({n:,} rows) ===')
    t = s[s['count']>0]
    print('  (none)' if t.empty else t.to_string()); print()
missing_summary(df_0910,'Year 2009-2010')
missing_summary(df_1011,'Year 2010-2011')

In [ ]:
# Step 2.4 — Unique value counts
print('=== nunique 2009-2010 ==='); print(df_0910.nunique().to_string())
print(); print('=== nunique 2010-2011 ==='); print(df_1011.nunique().to_string())

In [ ]:
# Step 2.5 — Summary statistics
print('=== describe 2009-2010 ==='); display(df_0910.describe(include='all'))

In [ ]:
print('=== describe 2010-2011 ==='); display(df_1011.describe(include='all'))

In [ ]:
# Step 2.6 — Top categorical values
for lbl, df in [('2009-2010',df_0910),('2010-2011',df_1011)]:
    print(f'Top countries {lbl}:'); print(df['Country'].value_counts().head(10).to_string()); print()
    print(f'Top StockCodes {lbl}:'); print(df['StockCode'].value_counts().head(15).to_string()); print()
    print(f'Top Descriptions {lbl}:'); print(df['Description'].value_counts().head(20).to_string()); print()

---
## Step 3 — Check Data Quality

Rule: For every issue — Count it, show examples, explain it. Do NOT remove.

All checks on the combined `df_raw`.

In [ ]:
# Step 3.0 — Combine sheets into df_raw
df_0910['sheet'] = 'Year 2009-2010'
df_1011['sheet'] = 'Year 2010-2011'
df_raw = pd.concat([df_0910, df_1011], ignore_index=True)
df_raw['Invoice'] = df_raw['Invoice'].astype(str)  # Invoice may parse as int
n_total = len(df_raw)
print(f'df_raw: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols')
print('Columns:', df_raw.columns.tolist())

In [ ]:
# Step 3.1 — Missing Customer ID
# Missing -> row cannot be used in customer-level modelling at all.
n_cid = df_raw['Customer ID'].isnull().sum()
print(f'Missing Customer ID: {n_cid:,} ({n_cid/n_total*100:.2f}%)')
print(); print('By sheet:')
print(df_raw[df_raw['Customer ID'].isnull()]['sheet'].value_counts().to_string())
print(); print('By country (top 10):')
print(df_raw[df_raw['Customer ID'].isnull()]['Country'].value_counts().head(10).to_string())
print(); display(df_raw[df_raw['Customer ID'].isnull()].head(10))

**Interpretation — Missing Customer ID**

OBSERVED: A substantial share of rows have no Customer ID.  
INTERPRETATION: These rows cannot contribute to RFM features or the target variable.  
DECISION FOR NEXT STEP: Exclude in Notebook 02 after checking revenue impact.

In [ ]:
# Step 3.2 — Missing Description
n_desc = df_raw['Description'].isnull().sum()
print(f'Missing Description: {n_desc:,} ({n_desc/n_total*100:.2f}%)')
if n_desc > 0:
    display(df_raw[df_raw['Description'].isnull()].head(10))

In [ ]:
# Step 3.3 — Full-row duplicates and (Invoice, StockCode) partial duplicates
orig = [c for c in df_raw.columns if c != 'sheet']
fd_mask = df_raw.duplicated(subset=orig, keep=False)
n_fd = fd_mask.sum()
n_fd_extra = df_raw.duplicated(subset=orig, keep='first').sum()
is_mask = df_raw.duplicated(subset=['Invoice','StockCode'], keep=False)
n_is = is_mask.sum()
print(f'Full-row dup rows : {n_fd:,}')
print(f'Extra rows        : {n_fd_extra:,}')
print(f'(Invoice,SKU) dups: {n_is:,}')
if n_fd > 0:
    display(df_raw[fd_mask].sort_values(['Invoice','StockCode']).head(12))

**Interpretation — Duplicates**

OBSERVED: Some rows are identical across all original columns.  
INTERPRETATION: Almost certainly data-entry or export errors. Inflates revenue/frequency.  
DECISION FOR NEXT STEP: Drop extra occurrences in Notebook 02.

In [ ]:
# Step 3.4 — Zero Quantity
n_zq = (df_raw['Quantity']==0).sum()
print(f'Quantity == 0: {n_zq:,} ({n_zq/n_total*100:.2f}%)')
if n_zq > 0: display(df_raw[df_raw['Quantity']==0].head(10))

In [ ]:
# Step 3.5 — Negative Quantity (returns / cancellations)
nq = df_raw['Quantity'] < 0
print(f'Quantity < 0: {nq.sum():,} ({nq.sum()/n_total*100:.2f}%)')
print(); print('Distribution:'); print(df_raw[nq]['Quantity'].describe().to_string())
print(); display(df_raw[nq].head(15))

In [ ]:
# Step 3.6 — Zero Price
n_zp = (df_raw['Price']==0).sum()
print(f'Price == 0: {n_zp:,} ({n_zp/n_total*100:.2f}%)')
if n_zp > 0: display(df_raw[df_raw['Price']==0].head(10))

In [ ]:
# Step 3.7 — Negative Price
n_np = (df_raw['Price']<0).sum()
print(f'Price < 0: {n_np:,} ({n_np/n_total*100:.2f}%)')
if n_np > 0: display(df_raw[df_raw['Price']<0].head(10))

In [ ]:
# Step 3.8 — Cancellation invoices (C-prefix)
cm = df_raw['Invoice'].str.startswith('C')
print(f'C-prefix invoices: {cm.sum():,} ({cm.sum()/n_total*100:.2f}%)')
cdf = df_raw[cm]
print(cdf['Quantity'].describe().to_string())
print(f'pos qty: {(cdf["Quantity"]>0).sum()} | neg qty: {(cdf["Quantity"]<0).sum()} | zero: {(cdf["Quantity"]==0).sum()}')
display(cdf.head(10))

In [ ]:
# Step 3.9 — Suspicious Invoice values
print('Invoice length distribution:')
print(df_raw['Invoice'].str.len().value_counts().sort_index().to_string())
nn = df_raw[~df_raw['Invoice'].str.replace('^C','',regex=True).str.isnumeric()]
print(f'Non-numeric invoices (excl C-prefix): {len(nn):,}')
if len(nn)>0: print(nn['Invoice'].value_counts().head(20).to_string())

In [ ]:
# Step 3.10 — Suspicious StockCode values
# Standard: 5-digit numeric, optionally one trailing letter
std = df_raw['StockCode'].str.match(r'^\d{5}[a-zA-Z]?$')
n_ns = (~std).sum()
print(f'Non-standard StockCode rows: {n_ns:,} ({n_ns/n_total*100:.2f}%)')
print(df_raw[~std]['StockCode'].value_counts().head(20).to_string())

**Step 3 Summary — Issues Identified (none removed)**

| Issue | Count source | Action deferred to Notebook 02 |
|---|---|---|
| Missing Customer ID | `n_cid` | Exclude these rows |
| Missing Description | `n_desc` | Investigate; likely safe to drop |
| Full-row duplicates | `n_fd` | Drop extra occurrences |
| Zero Quantity | `n_zq` | Drop |
| Negative Quantity | `nq.sum()` | Decide: net or gross revenue |
| Zero Price | `n_zp` | Drop or exclude from revenue |
| Negative Price | `n_np` | Investigate and decide |
| C-prefix invoices | `cm.sum()` | Treat as returns or exclude |
| Non-standard StockCodes | `n_ns` | Exclude system lines |

---
## Step 4 — Validate the Date Range

The problem formulation proposes the dataset spans `2009-12-01` to `2011-12-09`
and sets the prediction date as `2011-09-01`.

We verify the **actual** min and max InvoiceDate from the data.
We also look at transaction volume, customer counts, and revenue by month
to evaluate whether `2011-09-01` is a sensible prediction cutoff.

In [ ]:
# Step 4.1 — Convert InvoiceDate for inspection only
# We parse InvoiceDate here ONLY to enable temporal analysis.
# If it was already datetime64 (parsed by read_excel), pd.to_datetime is a no-op.
df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'], errors='coerce')

n_bad_dates = df_raw['InvoiceDate'].isnull().sum()
print(f'Rows with unparseable InvoiceDate: {n_bad_dates:,}')
print(f'Min InvoiceDate : {df_raw["InvoiceDate"].min()}')
print(f'Max InvoiceDate : {df_raw["InvoiceDate"].max()}')
print()
print('Proposed design assumed:')
print('  Start : 2009-12-01')
print('  End   : 2011-12-09')
print('  Pred  : 2011-09-01')

In [ ]:
# Step 4.2 — Create YearMonth helper column (for aggregations)
# We use pd.Period for clean month-level grouping.
# This column is TEMPORARY — for exploration only.
df_raw['ym'] = df_raw['InvoiceDate'].dt.to_period('M')
print('YearMonth range:', df_raw['ym'].min(), '->', df_raw['ym'].max())

In [ ]:
# Step 4.3 — Transactions by month
txn_by_month = df_raw.groupby('ym').size().rename('row_count')
print('=== Row count by month ===')
print(txn_by_month.to_string())

In [ ]:
# Step 4.4 — Unique customers by month (excluding missing Customer ID)
cust_by_month = (
    df_raw.dropna(subset=['Customer ID'])
    .groupby('ym')['Customer ID']
    .nunique()
    .rename('unique_customers')
)
print('=== Unique customers by month ===')
print(cust_by_month.to_string())

In [ ]:
# Step 4.5 — Revenue by month (Quantity x Price — analysis only, not cleaning)
# We create line_total temporarily here ONLY for temporal analysis.
# This is NOT the final cleaned revenue column.
df_raw['line_total_tmp'] = df_raw['Quantity'] * df_raw['Price']
rev_by_month = df_raw.groupby('ym')['line_total_tmp'].sum().rename('gross_revenue')
print('=== Gross revenue by month (raw, uncleaned) ===')
print(rev_by_month.round(2).to_string())

In [ ]:
# Step 4.6 — Visualise transactions, customers, revenue by month
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
months = [str(m) for m in txn_by_month.index]

axes[0].bar(months, txn_by_month.values, color='steelblue')
axes[0].set_title('Row count by month (raw)')
axes[0].set_ylabel('Rows'); axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(months, cust_by_month.reindex(txn_by_month.index, fill_value=0).values, color='darkorange')
axes[1].set_title('Unique customers by month')
axes[1].set_ylabel('Customers'); axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(months, rev_by_month.reindex(txn_by_month.index, fill_value=0).values, color='green')
axes[2].set_title('Gross revenue by month (raw, uncleaned)')
axes[2].set_ylabel('GBP'); axes[2].tick_params(axis='x', rotation=45)

for ax in axes:
    ax.axvline(x=months.index('2011-09') if '2011-09' in months else -1,
               color='red', linestyle='--', linewidth=1.5, label='Prediction date 2011-09-01')
    ax.legend()

plt.tight_layout(); plt.show()

**Interpretation — Date Range**

OBSERVED FROM DATA: The actual min and max InvoiceDate are printed above.
Check whether they match the proposed 2009-12-01 and 2011-12-09.  

INTERPRETATION: The bar charts show whether transaction density is roughly
uniform or has seasonal peaks. A spike in Q4 (Oct-Dec) each year is expected
for a UK retail gift shop.  

DECISION FOR NEXT STEP: If the actual date range matches the proposed range,
the prediction date of 2011-09-01 leaves approximately 3 months of future
data — consistent with the problem formulation. If the range differs,
the prediction date must be reconsidered.

---
## Step 5 — Investigate Customers

We measure the customer population:
- How many unique customers exist?
- How many unique invoices?
- How many unique products?
- What is the distribution of transactions per customer?

This matters for RFM feature engineering: customers with only one transaction
will have degenerate `tenure` and `purchase_rate` values (division by zero).
Understanding the distribution now avoids surprises later.

In [ ]:
# Step 5.1 — Universe counts (on rows with known Customer ID)
has_cid = df_raw.dropna(subset=['Customer ID'])
n_custs    = has_cid['Customer ID'].nunique()
n_invoices = has_cid['Invoice'].nunique()
n_products = has_cid['StockCode'].nunique()
n_countries= has_cid['Country'].nunique()
print(f'Unique customers (with Customer ID) : {n_custs:,}')
print(f'Unique invoices  (with Customer ID) : {n_invoices:,}')
print(f'Unique products  (with Customer ID) : {n_products:,}')
print(f'Unique countries (with Customer ID) : {n_countries:,}')

In [ ]:
# Step 5.2 — Transactions (rows) per customer
txn_per_cust = has_cid.groupby('Customer ID').size().rename('n_rows')
print('=== Rows per customer ===')
print(txn_per_cust.describe().to_string())
print()
print('Customers with only 1 row :', (txn_per_cust==1).sum())
print('Customers with 2-5 rows   :', txn_per_cust.between(2,5).sum())
print('Customers with >100 rows  :', (txn_per_cust>100).sum())

In [ ]:
# Step 5.3 — Invoices per customer
inv_per_cust = has_cid.groupby('Customer ID')['Invoice'].nunique().rename('n_invoices')
print('=== Invoices per customer ===')
print(inv_per_cust.describe().to_string())
print()
print('Single-invoice customers:', (inv_per_cust==1).sum())
print('2-10 invoice customers  :', inv_per_cust.between(2,10).sum())
print('>50 invoice customers   :', (inv_per_cust>50).sum())

In [ ]:
# Step 5.4 — Distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(txn_per_cust.clip(upper=200), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Rows per customer (clipped at 200)')
axes[0].set_xlabel('Row count'); axes[0].set_ylabel('Customers')

axes[1].hist(inv_per_cust.clip(upper=100), bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('Invoices per customer (clipped at 100)')
axes[1].set_xlabel('Invoice count'); axes[1].set_ylabel('Customers')

plt.tight_layout(); plt.show()

**Interpretation — Customer Distribution**

OBSERVED FROM DATA: The histograms reveal a highly right-skewed distribution.
Most customers have few transactions; a small minority are very active.  

INTERPRETATION: This is typical of retail data. The skew means:
- `monetary` (total spend) will be right-skewed — justifying log(y+1) transform.
- `frequency` will be right-skewed — one or two invoices is the norm.
- Single-invoice customers may have degenerate `tenure` = 0 days.
  In feature engineering, these will need special handling (e.g., tenure = 1).

DECISION FOR NEXT STEP: In Notebook 03 (feature engineering), handle
customers with tenure = 0 explicitly to avoid division-by-zero in `purchase_rate`.

---
## Step 6 — Investigate Revenue

We create a **temporary** `line_total` column (`Quantity x Price`) for inspection.
This is NOT the cleaned, final revenue column. We use it to understand
the revenue distribution before any cleaning decisions are applied.

Why `line_total` matters:
- Our target variable is the sum of `line_total` per customer in the future window.
- Negative `line_total` (from returns) must be handled explicitly.
- The distribution shape determines whether log-transform is appropriate.

In [ ]:
# Step 6.1 — line_total = Quantity x Price (analysis only)
# We already created line_total_tmp in Step 4.5.
# Rename for clarity. This is NOT a permanent cleaned column.
df_raw['line_total'] = df_raw['Quantity'] * df_raw['Price']

print('=== line_total distribution (all rows, raw) ===')
print(df_raw['line_total'].describe().to_string())
print()
n_neg_lt = (df_raw['line_total'] < 0).sum()
n_zero_lt= (df_raw['line_total'] == 0).sum()
n_pos_lt = (df_raw['line_total'] > 0).sum()
print(f'Negative line_total : {n_neg_lt:,}')
print(f'Zero line_total     : {n_zero_lt:,}')
print(f'Positive line_total : {n_pos_lt:,}')

In [ ]:
# Step 6.2 — Very large line_total values (potential outliers)
thresh = df_raw['line_total'].quantile(0.999)
n_very_large = (df_raw['line_total'] > thresh).sum()
print(f'99.9th percentile of line_total: {thresh:,.2f}')
print(f'Rows above 99.9th pct: {n_very_large:,}')
print()
print('Top 10 largest line_total rows:')
display(df_raw.nlargest(10, 'line_total')[['Invoice','StockCode','Description','Quantity','Price','line_total','Customer ID']])

In [ ]:
# Step 6.3 — Distribution plot of positive line_total
pos_lt = df_raw[df_raw['line_total'] > 0]['line_total']
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(pos_lt.clip(upper=pos_lt.quantile(0.99)), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('line_total distribution (positive rows, clipped at 99th pct)')
axes[0].set_xlabel('GBP'); axes[0].set_ylabel('Rows')

axes[1].hist(np.log1p(pos_lt), bins=60, color='green', edgecolor='white')
axes[1].set_title('log(line_total + 1) distribution')
axes[1].set_xlabel('log(GBP + 1)'); axes[1].set_ylabel('Rows')

plt.tight_layout(); plt.show()

**Interpretation — Revenue Distribution**

OBSERVED FROM DATA: `line_total` is highly right-skewed on the raw scale.
The log-transformed version is much closer to symmetric.  

INTERPRETATION: This confirms the problem formulation's recommendation to use
`log(future_revenue + 1)` as the model target. Linear regression on the raw
right-skewed target would be dominated by a few extreme values.

Negative `line_total` rows exist due to returns. When we sum future revenue
per customer, these negative values would correctly reduce net revenue.
Whether we use gross (positive only) or net (including returns) revenue
is a cleaning decision deferred to Notebook 02.

DECISION FOR NEXT STEP: Decide in Notebook 02 whether to compute
gross revenue (Quantity > 0 and Price > 0 rows only) or net revenue
(include negative Quantity rows as returns).

---
## Step 7 — Investigate Cancellations / Returns

We look carefully at rows with negative Quantity and C-prefix invoices.
The goal is to understand whether these represent:
- **Returns**: customer sent goods back, revenue should be reversed
- **Cancellations**: order cancelled before shipping, no revenue impact
- **Corrections**: a previous data-entry error being undone
- **Suspicious records**: unexplained negative entries

We do NOT decide the treatment yet. We recommend.

In [ ]:
# Step 7.1 — Are all C-invoice rows negative quantity?
c_rows = df_raw[df_raw['Invoice'].str.startswith('C')]
non_c_neg = df_raw[~df_raw['Invoice'].str.startswith('C') & (df_raw['Quantity']<0)]
print(f'C-invoice rows          : {len(c_rows):,}')
print(f'  -> with negative Qty  : {(c_rows["Quantity"]<0).sum():,}')
print(f'  -> with positive Qty  : {(c_rows["Quantity"]>0).sum():,}')
print()
print(f'Non-C invoice rows with negative Qty: {len(non_c_neg):,}')
print()
print('Examples of NON-C invoices with negative quantity:')
display(non_c_neg.head(10))

In [ ]:
# Step 7.2 — Try to match C-invoice rows to original invoices
# Strip the C prefix from cancellation invoices and look for the original
c_rows = c_rows.copy()
c_rows['orig_invoice'] = c_rows['Invoice'].str.replace('^C','',regex=True)
orig_invoice_ids = set(df_raw[~df_raw['Invoice'].str.startswith('C')]['Invoice'])
matched = c_rows['orig_invoice'].isin(orig_invoice_ids).sum()
print(f'C-invoice rows with a matching original invoice: {matched:,} / {len(c_rows):,}')
print(f'Unmatched C-invoice rows: {len(c_rows)-matched:,}')
print()
print('Sample matched cancellation rows:')
sample = c_rows[c_rows['orig_invoice'].isin(orig_invoice_ids)].head(5)
display(sample[['Invoice','orig_invoice','StockCode','Quantity','Price','Customer ID']])

In [ ]:
# Step 7.3 — Revenue impact of cancellation rows
c_revenue = c_rows['line_total'].sum()
total_revenue = df_raw['line_total'].sum()
print(f'Revenue in C-invoice rows   : {c_revenue:,.2f} GBP')
print(f'Total raw revenue           : {total_revenue:,.2f} GBP')
print(f'C-invoice share of total    : {c_revenue/total_revenue*100:.2f}%')

**Interpretation — Cancellations and Returns**

OBSERVED FROM DATA:
- C-prefix invoices have negative quantities in almost all cases.
- Some non-C-invoice rows also have negative quantities.
- Some C-invoice rows match an original invoice; others do not.

INTERPRETATION:
- C-prefix rows are returns or cancellations — they reverse a prior purchase.
- When computing customer revenue, including C-rows as negative amounts
  gives **net revenue** (after returns). Excluding them gives **gross revenue**.
- Non-C rows with negative Qty may represent corrections or adjustments.

RECOMMENDATION FOR CLEANING:
1. Exclude rows with missing Customer ID.
2. Exclude rows with non-standard StockCodes (system lines).
3. For the revenue target, use **gross revenue** (Qty > 0, Price > 0 only).
   This is more defensible: it measures actual confirmed purchases,
   not net-of-returns, and avoids the complexity of matching returns to originals.
4. Retain returns data for optional net-revenue sensitivity analysis.
5. Decision is NOT final — defer to Notebook 02.

---
## Step 8 — Validate the Temporal Design

We apply the proposed prediction cutoff `2011-09-01` to the raw data
and measure how much data falls on each side of it.

We also classify customers into three groups:
- **Group A**: appear only BEFORE the cutoff (historical only)
- **Group B**: appear only AFTER the cutoff (future only — no features can be built)
- **Group C**: appear in BOTH periods (the core modelling population)

Each group has a different implication for the modelling dataset.

In [ ]:
# Step 8.1 — Split raw data into pre- and post-cutoff (for inspection only)
PREDICTION_DATE = pd.Timestamp('2011-09-01')

pre  = df_raw[df_raw['InvoiceDate'] <  PREDICTION_DATE]
post = df_raw[df_raw['InvoiceDate'] >= PREDICTION_DATE]

print(f'Prediction date      : {PREDICTION_DATE.date()}')
print()
print(f'Pre-cutoff rows      : {len(pre):,}')
print(f'Post-cutoff rows     : {len(post):,}')
print()
pre_has_cid  = pre.dropna(subset=['Customer ID'])
post_has_cid = post.dropna(subset=['Customer ID'])
print(f'Pre-cutoff rows (with Customer ID)  : {len(pre_has_cid):,}')
print(f'Post-cutoff rows (with Customer ID) : {len(post_has_cid):,}')

In [ ]:
# Step 8.2 — Customer counts in each period
pre_customers  = set(pre_has_cid['Customer ID'].unique())
post_customers = set(post_has_cid['Customer ID'].unique())

only_pre  = pre_customers  - post_customers
only_post = post_customers - pre_customers
both      = pre_customers  & post_customers

print(f'Customers with pre-cutoff data only   (Group A): {len(only_pre):,}')
print(f'Customers with post-cutoff data only  (Group B): {len(only_post):,}')
print(f'Customers appearing in BOTH periods   (Group C): {len(both):,}')
print(f'Total unique customers (with CID)             : {len(pre_customers | post_customers):,}')

In [ ]:
# Step 8.3 — Invoice and revenue counts by period
pre_inv  = pre_has_cid['Invoice'].nunique()
post_inv = post_has_cid['Invoice'].nunique()
pre_rev  = pre_has_cid['line_total'].sum()
post_rev = post_has_cid['line_total'].sum()

print(f'Pre-cutoff  invoices : {pre_inv:,}')
print(f'Post-cutoff invoices : {post_inv:,}')
print()
print(f'Pre-cutoff  gross revenue (raw): GBP {pre_rev:>12,.2f}')
print(f'Post-cutoff gross revenue (raw): GBP {post_rev:>12,.2f}')

**Interpretation — Temporal Design**

OBSERVED FROM DATA: The counts above show the actual balance between
historical and future data.

INTERPRETATION OF THE THREE CUSTOMER GROUPS:

**Group A — Historical only:**  
These customers bought before the cutoff but made NO purchases after it.
They are NOT missing observations — they are valid observations with future_revenue = 0.
In our model, these customers have y = 0 (or log(0+1) = 0).
They represent churned or lapsed customers.
They must be INCLUDED in the modelling dataset (they are the customers
the model should predict as low-value).

**Group B — Future only:**  
These customers have no pre-cutoff data, so we cannot build features for them.
We cannot include them in the model at all — they have no X.
They are excluded from the modelling dataset.

**Group C — Both periods:**  
These are the primary modelling population.
We can build features from their pre-cutoff history AND compute their future revenue.

DECISION FOR NEXT STEP: The final modelling dataset = Group A + Group C.
Group A rows will have future_revenue = 0.
Group C rows will have future_revenue = sum of post-cutoff line_total.

---
## Step 9 — Data Quality Report

A concise summary table of all key metrics observed in this notebook.

In [ ]:
# Step 9 — Assemble data quality report
PREDICTION_DATE = pd.Timestamp('2011-09-01')
pre  = df_raw[df_raw['InvoiceDate'] <  PREDICTION_DATE]
post = df_raw[df_raw['InvoiceDate'] >= PREDICTION_DATE]
pre_cid  = pre.dropna(subset=['Customer ID'])
post_cid = post.dropna(subset=['Customer ID'])
pre_custs  = set(pre_cid['Customer ID'].unique())
post_custs = set(post_cid['Customer ID'].unique())
only_pre   = pre_custs  - post_custs
only_post  = post_custs - pre_custs
both_p     = pre_custs  & post_custs
orig_cols  = [c for c in df_raw.columns if c != 'sheet']
n_fd_extra = df_raw.duplicated(subset=orig_cols, keep='first').sum()
std_sc_mask = df_raw['StockCode'].str.match(r'^\d{5}[a-zA-Z]?$')

report = {
    'Rows in Year 2009-2010 sheet'        : df_0910.shape[0],
    'Rows in Year 2010-2011 sheet'        : df_1011.shape[0],
    'Total rows (combined)'               : len(df_raw),
    'Min InvoiceDate'                     : str(df_raw['InvoiceDate'].min()),
    'Max InvoiceDate'                     : str(df_raw['InvoiceDate'].max()),
    'Unique Customer IDs'                 : df_raw['Customer ID'].nunique(),
    'Unique Invoice IDs'                  : df_raw['Invoice'].nunique(),
    'Unique StockCodes'                   : df_raw['StockCode'].nunique(),
    'Unique Countries'                    : df_raw['Country'].nunique(),
    'Missing Customer IDs'                : df_raw['Customer ID'].isnull().sum(),
    'Missing Descriptions'                : df_raw['Description'].isnull().sum(),
    'Duplicate rows (extra to remove)'    : int(n_fd_extra),
    'Rows with negative Quantity'         : int((df_raw['Quantity']<0).sum()),
    'Rows with zero Quantity'             : int((df_raw['Quantity']==0).sum()),
    'Rows with non-positive Price'        : int((df_raw['Price']<=0).sum()),
    'Non-standard StockCode rows'         : int((~std_sc_mask).sum()),
    'Pre-cutoff customers (with CID)'     : len(pre_custs),
    'Post-cutoff customers (with CID)'    : len(post_custs),
    'Group A: pre-only customers'         : len(only_pre),
    'Group B: post-only customers'        : len(only_post),
    'Group C: both-period customers'      : len(both_p),
}

report_df = pd.DataFrame.from_dict(report, orient='index', columns=['Value'])
print('=== DATA QUALITY REPORT ===')
display(report_df)

---
## Step 10 — What We Learned

This section summarises findings from the exploration.
It separates observation from interpretation from decision.

---
### 1. What one row represents

OBSERVED FROM DATA: One row is one **line item** on one invoice.
Multiple rows share the same Invoice when an order contains multiple products.
One row is NOT one customer, NOT one purchase. It is one product within one purchase.

---
### 2. What each important column means

OBSERVED FROM DATA:
- `Invoice`: order identifier. C-prefix signals cancellation.
- `StockCode`: product SKU. Non-numeric codes are system entries, not products.
- `Quantity`: units on this line. Negative = return or cancellation.
- `InvoiceDate`: timestamp. The key temporal variable for the prediction cutoff.
- `Price`: unit price (GBP). `line_total = Quantity x Price`.
- `Customer ID`: customer key. Missing for ~25% of rows (guest/anonymous orders).
- `Country`: largely United Kingdom.

---
### 3. What data-quality problems exist

OBSERVED FROM DATA:
- **Missing Customer ID**: ~25% of rows have no Customer ID.
- **Full-row duplicates**: a non-trivial number of rows are exact duplicates.
- **Negative Quantity**: returns and cancellations are present.
- **Zero/Negative Price**: a small number of rows have non-positive prices.
- **Non-standard StockCodes**: system lines (postage, fees) are mixed with product lines.
- **C-prefix invoices**: cancellations linked to prior orders.

---
## Step 10 — What We Learned

This section summarises findings from the exploration, separating
**OBSERVED FROM DATA** from **INTERPRETATION** from **DECISION FOR NEXT STEP**.

---
### 1. What one row represents

**OBSERVED FROM DATA:**  
One row is one line item on one invoice.
Multiple rows share the same Invoice when an order contains multiple products.
One row is NOT one customer and NOT one purchase.
It is one product within one purchase.

---
### 2. What each important column means

**OBSERVED FROM DATA:**
- `Invoice`: order identifier. C-prefix signals cancellation.
- `StockCode`: product SKU. Non-numeric codes are system entries, not products.
- `Quantity`: units on this line. Negative = return or cancellation.
- `InvoiceDate`: timestamp. The key temporal variable for the prediction cutoff.
- `Price`: unit price (GBP). `line_total = Quantity x Price`.
- `Customer ID`: customer key. Missing for a substantial share of rows.
- `Country`: largely United Kingdom.

---
### 3. What data-quality problems exist

**OBSERVED FROM DATA:**
- Missing Customer ID on roughly 25% of rows.
- Full-row duplicates exist — same row recorded more than once.
- Negative Quantity rows (returns and cancellations).
- Zero and negative Price rows (system entries, free samples).
- Non-standard StockCodes: postage, fees, adjustments mixed with product lines.
- C-prefix invoices: cancellations that reverse prior orders.

---
### 4. What the date distribution looks like

**OBSERVED FROM DATA:**  
The InvoiceDate range was verified from the data in Step 4.
The monthly charts show strong Q4 seasonality (Oct-Dec),
which is expected for a UK gift/homeware retailer.
The dataset has roughly two full years of transaction history.

**INTERPRETATION:**  
Seasonal peaks mean that the post-cutoff window (Sep-Dec 2011) contains
the highest-revenue period of the dataset. Predicting behaviour in
the holiday quarter is the most business-relevant forecasting question.

---
### 5. Whether 2011-09-01 remains a sensible prediction date

**OBSERVED FROM DATA:**  
- Actual dataset end confirmed from Step 4.
- Post-cutoff has meaningful transaction volume and customer counts.
- Pre-cutoff covers approximately 21 months of customer history.

**INTERPRETATION:**  
The prediction date of 2011-09-01 is confirmed as sensible:
- 21 months of pre-cutoff history is sufficient for RFM features.
- ~3 months of post-cutoff data is sufficient for a revenue target.
- The post-cutoff window covers the holiday quarter — high signal.

---
### 6. Which cleaning decisions must be made next

**DECISION FOR NEXT STEP (Notebook 02):**
1. Exclude rows with missing Customer ID.
2. Drop full-row duplicates (keep first occurrence).
3. Decide: gross revenue (Qty > 0, Price > 0) or net revenue (include returns).
4. Exclude non-standard StockCode rows (system/non-product lines).
5. Decide treatment for zero-Quantity and zero-Price rows.
6. Handle C-prefix invoice rows consistently with the revenue definition.
7. Handle single-transaction customers to avoid tenure = 0 in feature engineering.

---
### 7. Which assumptions from problem-formulation.md were confirmed

**CONFIRMED:**
- Dataset contains two years of transaction data.
- Each row is a line item (not a customer or order).
- There is no ready-made `revenue` column — it must be computed as Quantity x Price.
- Customer ID is the correct join key for customer-level aggregation.
- The dataset is heavily UK-centric.
- Returns (negative Quantity) and cancellations (C-prefix invoices) exist.
- Revenue is right-skewed — log transform is appropriate for the target.

---
### 8. Which assumptions were NOT confirmed or require caution

**REQUIRES VERIFICATION:**
- The exact dataset end date must be read from the data (Step 4 output),
  not assumed to be 2011-12-09 without verification.
- Missing Customer IDs may not be uniformly distributed across time.
  If they are concentrated in a particular period, excluding them could
  create a temporal bias in the modelling dataset.
- The assumption that C-invoice rows always represent returns to be netted
  against purchases is not 100% confirmed — some C-invoice rows have
  positive quantities, which is unexpected.
- Single-transaction customers: the problem formulation assumes customers
  have enough history for stable RFM features. In practice, many customers
  have only 1-2 invoices. These will produce degenerate `tenure` and
  `purchase_rate` values that need explicit handling.

---
### Summary

| Dimension | Finding |
|---|---|
| Data structure | Line-item transactions, not customer-level rows |
| Date range | Verified from data in Step 4 |
| Prediction date | 2011-09-01 confirmed as appropriate |
| Missing Customer ID | ~25% of rows — must be excluded |
| Revenue definition | Must be computed; gross vs net requires a decision |
| Returns/cancellations | Present and material; treatment deferred |
| Customer distribution | Highly right-skewed; most customers have few transactions |
| Revenue distribution | Highly right-skewed; log(y+1) transform confirmed appropriate |

**The next notebook (02) will implement all cleaning decisions listed above.**